In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

import numpy as np

features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

In [2]:
from training_info import TrainingInfo
import pandas as pd

experiment_name = "smeared_fvt_training_ensemble"
TrainingInfo.update_metadata()
metadata = TrainingInfo.load_metadata()

hashes = TrainingInfo.find({
    "experiment_name": experiment_name, 
    "dataset": lambda x: x["seed"] >= 50
                   })


hparams_df = pd.DataFrame([
    {"hash": hash_, 
     "seed": metadata[hash_]["dataset"]["seed"], 
     "signal_ratio": metadata[hash_]["dataset"]["signal_ratio"], 
     "n_3b": metadata[hash_]["dataset"]["n_3b"], 
     "ratio_4b": metadata[hash_]["dataset"]["ratio_4b"], 
     "signal_filename": metadata[hash_]["dataset"]["signal_filename"], 
     "train_seed": metadata[hash_]["train_seed"],
     "noise_scale": metadata[hash_]["smearing"]["noise_scale"],
    }
    for hash_ in hashes
])

2025-08-26 23:03:48,961 - INFO - Adding 1200 files, removing 0 hashes
100%|██████████| 1200/1200 [00:02<00:00, 401.63it/s]


In [3]:
hashes = TrainingInfo.find({
        "experiment_name": "CR_fvt_training_ensemble_max", 
        "dataset": lambda x: (x["seed"] >= 50),
    })
len(hashes)

3500

In [5]:
counts = (hparams_df.groupby(["train_seed", "signal_ratio", "noise_scale"])["hash"].count() != 50)
corrupted_hyperparams = counts[counts].reset_index()[["train_seed", "signal_ratio", "noise_scale"]].values

corrupted_hashes = []
for train_seed, signal_ratio, noise_scale in corrupted_hyperparams:
    new_hashes = TrainingInfo.find({
        "experiment_name": "smeared_fvt_training_ensemble", 
        "train_seed": train_seed,
        "dataset": lambda x: (x["seed"] >= 50)
                              and (x["signal_ratio"] == signal_ratio),
        "smearing": lambda x: (x["noise_scale"] == noise_scale)
    })
    print(len(new_hashes))
    corrupted_hashes.extend(new_hashes)

len(corrupted_hashes)

0

In [ ]:
# counts = (hparams_df.groupby(["train_seed", "signal_ratio", "noise_scale"])["hash"].count() != 50)
# corrupted_hyperparams = counts[counts].reset_index()[["train_seed", "signal_ratio", "noise_scale"]].values

# corrupted_hashes = []
# for train_seed, signal_ratio, noise_scale in corrupted_hyperparams:
#     new_hashes = TrainingInfo.find({
#         "experiment_name": "smeared_fvt_training_ensemble", 
#         "train_seed": train_seed,
#         "dataset": lambda x: (x["seed"] >= 50)
#                               and (x["signal_ratio"] == signal_ratio),
#         "smearing": lambda x: (x["noise_scale"] == noise_scale)
#     })
#     print(len(new_hashes))
#     corrupted_hashes.extend(new_hashes)

# len(corrupted_hashes)

# TrainingInfo.delete(corrupted_hashes)

33
25
Deleting 58 hashes


2025-08-22 00:06:58,688 - INFO - Adding 0 files, removing 58 hashes
0it [00:00, ?it/s]


In [18]:
dupl = hparams_df.set_index(["seed", "train_seed", "signal_ratio", "noise_scale"])
duplicated_hashes = dupl.loc[dupl.index.duplicated(), "hash"].values

In [19]:
len(duplicated_hashes)

0

In [ ]:
# TrainingInfo.delete(duplicated_hashes)

Deleting 50 hashes


2025-08-22 00:00:53,179 - INFO - Adding 0 files, removing 50 hashes
0it [00:00, ?it/s]


In [ ]:
# # fetch all filenames in data/checkpoints/

# import glob

# filenames = glob.glob("/home/export/soheuny/SRFinder/soheun/data/checkpoints/*")

100925